# 第 8 章 · 记忆（LangChain / LangGraph）

四步演进，正好是"记忆"这件事在 LangChain 里的搬家史：

1. `ChatMessageHistory` —— 就是一个消息列表
2. `ConversationBufferMemory` —— 把它包装成能塞进 prompt 的变量
3. `LLMChain` + memory —— 链条自动读写记忆（v0.3 老 API，已搬到 `langchain_classic`）
4. LangGraph `BaseStore` —— 记忆**跨会话**存在 store 里，还能让 agent **改写自己的指令**

kernel 选 `Python (agentic-book)`。

In [1]:
# 【适配】common 已由 venv 的启动引导加进 sys.path（见 tools/_book_bootstrap.py）
from common import make_llm

# 【适配】LangChain 1.x 把 v0.3 的 memory / chains 搬到了 langchain_classic
from langchain_classic.memory import ChatMessageHistory, ConversationBufferMemory
from langchain_classic.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.store.memory import InMemoryStore


In [2]:
# ============ ① ChatMessageHistory：就是一个消息列表 ============
history = ChatMessageHistory()
history.add_user_message("I'm heading to New York next week.")
history.add_ai_message("Great! It's a fantastic city.")

print("① 消息条数:", len(history.messages))
for m in history.messages:
    print(f"   [{m.type}] {m.content}")


① 消息条数: 2
   [human] I'm heading to New York next week.
   [ai] Great! It's a fantastic city.


/tmp/ipykernel_1805620/45501078.py:2: LangChainDeprecationWarning: The class `InMemoryChatMessageHistory` was deprecated in LangChain 1.6.4 and will be removed in 2.0.0 See the short-term memory documentation for recommended alternatives: https://docs.langchain.com/oss/python/langchain/short-term-memory
  history = ChatMessageHistory()


In [3]:
# ============ ② ConversationBufferMemory：能被塞进 prompt 的变量 ============
memory = ConversationBufferMemory()
memory.save_context({"input": "What's the weather like?"}, {"output": "It's sunny today."})

print("② memory_key =", memory.memory_variables)
print("② load_memory_variables ->", memory.load_memory_variables({}))


② memory_key = ['history']
② load_memory_variables -> {'history': "Human: What's the weather like?\nAI: It's sunny today."}


/tmp/ipykernel_1805620/740136429.py:2: LangChainDeprecationWarning: The class `ConversationBufferMemory` was deprecated in LangChain 0.3.1 and will be removed in 2.0.0. Use `langchain.agents.create_agent` instead. For agents that need to remember prior interactions, use `create_agent` with checkpointing or the `Store` API. See https://docs.langchain.com/oss/python/langchain/short-term-memory and https://docs.langchain.com/oss/python/langchain/long-term-memory
  memory = ConversationBufferMemory()


In [4]:
# ============ ③ LLMChain + memory：链条自动读写 ============
# 【适配】书里用 OpenAI(...)（completion 接口）；DeepSeek 只有 chat 接口，改用 ChatOpenAI
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel agent. Previous conversation:\n{history}"),
    ("human", "{question}"),
])
memory3 = ConversationBufferMemory(memory_key="history", return_messages=False)
conversation = LLMChain(llm=ChatOpenAI(temperature=0, model="deepseek-flash"),
                        prompt=prompt, memory=memory3)

print("③ 第 1 问 ->", conversation.invoke({"question": "我第一次去纽约，有什么推荐？"})["text"][:120])
print("③ 第 2 问（不带历史，靠 memory）->",
      conversation.invoke({"question": "我刚才说要去哪？"})["text"][:120])


/tmp/ipykernel_1805620/1023961384.py:8: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  conversation = LLMChain(llm=ChatOpenAI(temperature=0, model="deepseek-flash"),


③ 第 1 问 -> 第一次去纽约，建议不要贪多，按“区域”玩最省时间。纽约核心体验可以分成：经典地标、街区漫步、博物馆、百老汇、美食。下面给你一个比较适合第一次去的推荐框架。

## 第一次去最推荐的地方

**经典地标**
- **自由女神像 + 炮台公园*


③ 第 2 问（不带历史，靠 memory）-> 你刚才说：**你第一次去纽约**，想让我推荐玩法。


In [5]:
# ============ ④ LangGraph Store：跨会话的"程序记忆" ============
store = InMemoryStore()
store.put(("instructions",), "agent", {"instructions": "You are a helpful travel agent."})

prompt_template = """当前指令：
{instructions}

最近的对话：
{conversation}

请改进这段指令，让它更具体、更有用。只输出新的指令本身，不要解释。"""


def update_instructions(state, store):
    """读当前指令 + 近期对话 → 让模型改写 → 存回 store。"""
    namespace = ("instructions",)
    current = store.search(namespace)[0]
    prompt = prompt_template.format(
        instructions=current.value["instructions"],
        conversation=state["messages"],
    )
    new_instructions = make_llm(temperature=0.3).invoke(prompt).content.strip()
    store.put(namespace, "agent", {"instructions": new_instructions})
    return {"instructions": new_instructions}


In [6]:
# ---- ④ 执行：改写前后对比 ----
state = {"messages": ["用户想去纽约，喜欢靠窗座位", "我推荐了靠窗的酒店和中央公园"]}

before = store.get(("instructions",), "agent").value["instructions"]
result = update_instructions(state, store)
after = store.get(("instructions",), "agent").value["instructions"]

print("④ 改写前的指令:", before)
print("④ 改写后的指令:", after[:200])
print("④ 确实存回 store:", after == result["instructions"])


④ 改写前的指令: You are a helpful travel agent.
④ 改写后的指令: 你是一名专业的纽约旅行规划师。请根据用户偏好提供个性化、可执行的旅行建议：用户计划前往纽约且偏好靠窗体验，请先确认“靠窗”是指航班座位还是酒店房间视野，并询问出行日期、预算、同行人数和兴趣；随后推荐合适的航班座位、带景观的酒店及以中央公园为代表的景点，说明推荐理由、位置、交通方式和预订注意事项，避免笼统建议。
④ 确实存回 store: True
